# Quality control (QC) checks on raw data

Basic QC checks on the raw data from each gauge, including:

* Range checks, based on parameter- and location-specific limits
* Rate-of-change checks
* Persistence checks (static readings)

QC flags:
| Flag | Category | Description                         |
|:-----|:---------|:------------------------------------|
| 1    | Good     | Quality control check(s) passed     |
| 2    | Suspect  | Suspect data                        |
| 3    | Bad      | Not physically plausible            |


QC labels:
| Label | Description                   |
|:-----|:-------------------------------|
| A    | Above feasible range           |
| B    | Below feasible range           |
| C    | Low SNR                        |
| D    | High rate of change            |
| E    | Persistently constant value    |


In [ ]:
# Imports
import os
from pathlib import Path
import pandas as pd
import numpy as np
from typing import NamedTuple, List
from enum import Enum, unique
import re
import matplotlib.pyplot as plt

from blue_heart_data_handling import QCFlags, QCLabels, DataTypeOptions, get_gauge_data

In [ ]:
# General settings

# Whether to save plots
save_flag = True

# Gauge index
path_index_gauge_data = '../index_gauge_data.csv'
df_index = pd.read_csv(path_index_gauge_data)

# Folder to save plots in
path_output_folder = 'outputs/4_raw_data_quality_checks'
if not os.path.isdir(path_output_folder):
    os.makedirs(path_output_folder)

# Plot settings
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 10,
    'axes.titlesize': 10,
    'axes.labelsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10
})

## Quality control checks

In [ ]:
# Quality code consists of flag and label
QualityCode = NamedTuple('QualityCode', [
    ('Flag', QCFlags),
    ('Label', QCLabels)
])

### Define all static thresholds for range checks and function for checks

In [ ]:
# Define all static range thresholds

# Threshold types: Upper or lower range
@unique
class ThresholdType(Enum):
    Max = 'Max'
    Min = 'Min'

# Parameters to define a range check
RangeCheckParams = NamedTuple('RangeCheckParams', [
    ('Type', ThresholdType),
    ('Threshold', float),
    ('Flag', QCFlags),
    ('Label', QCLabels),
])

# All static thresholds to be applied for each parameter type
range_checks_all = {
    'Water Level [mAOD]': [],       # No static thresholds, as based on location-specific bank levels
    'Signal-to-noise ratio [dB]': [
        RangeCheckParams(Type=ThresholdType.Min, Threshold=10, Flag=QCFlags.Suspect, Label=QCLabels.LowSNR),  # SNR below 10dB
    ],
    'Water Depth [m]': [
        RangeCheckParams(Type=ThresholdType.Min, Threshold=0, Flag=QCFlags.Bad, Label=QCLabels.BelowRange),  # Water depth below 0
    ],
    'Water Level [mm]': [
        RangeCheckParams(Type=ThresholdType.Min, Threshold=0, Flag=QCFlags.Bad, Label=QCLabels.BelowRange),  # Water depth below 0
    ],
    'Sewer Level [mm]': [
        RangeCheckParams(Type=ThresholdType.Min, Threshold=0, Flag=QCFlags.Suspect, Label=QCLabels.BelowRange),  # Sewer level below 0
    ],
    'Depth To Water [m]': [
        RangeCheckParams(Type=ThresholdType.Min, Threshold=0, Flag=QCFlags.Bad, Label=QCLabels.BelowRange),  # Depth to water below 0
    ],
    'Rainfall Depth [mm]': [
        RangeCheckParams(Type=ThresholdType.Min, Threshold=0, Flag=QCFlags.Bad, Label=QCLabels.BelowRange),  # Rainfall depth below 0
        RangeCheckParams(Type=ThresholdType.Max, Threshold=50, Flag=QCFlags.Bad, Label=QCLabels.AboveRange),  # Rainfall depth above 50mm (in single, 15 min timestep)
    ],
    'Water Temperature [°C]': [
        RangeCheckParams(Type=ThresholdType.Min, Threshold=-2, Flag=QCFlags.Bad, Label=QCLabels.BelowRange),  # Water temperature below -0.5°C
        RangeCheckParams(Type=ThresholdType.Max, Threshold=35, Flag=QCFlags.Bad, Label=QCLabels.AboveRange)     # Water temperature above 35°C
    ],
    'Air Temperature [°C]': [
        RangeCheckParams(Type=ThresholdType.Min, Threshold=-20, Flag=QCFlags.Bad, Label=QCLabels.BelowRange),   # Air temperature below -20°C
        RangeCheckParams(Type=ThresholdType.Max, Threshold=50, Flag=QCFlags.Bad, Label=QCLabels.AboveRange)     # Air temperature above 50°C
    ],
}

In [ ]:
# Function to check values are within all relevant range thresholds for a given parameter

def quality_check_parameter_ranges(
    gauge_parameter: str,         # Name of gauge parameter
    df_parameter: pd.DataFrame,   # Dataframe containing data for a single parameter
    range_checks_all: dict,       # Dict containing list of RangeCheckParams for each gauge parameter,
    bed_level_estimate: float,
    bank_level_estimate: float
):
    # Initialise storage of all failure flags for current parameter (one dataframe per range check)
    dfs_failure_flags = []

    # Find details of ranges to check for given parameter (list of RangeCheckParams)
    range_checks_parameter = range_checks_all[gauge_parameter].copy()

    # If bank and bed levels exist and current parameter is 'Water Level [mAOD]', define additional range RangeChecks to check based on these
    if (not np.isnan(bed_level_estimate)) and (gauge_parameter == 'Water Level [mAOD]'):
        range_checks_parameter.append(RangeCheckParams(Type=ThresholdType.Min, Threshold=bed_level_estimate, Flag=QCFlags.Suspect, Label=QCLabels.BelowRange))         # Below estimated bed level
        range_checks_parameter.append(RangeCheckParams(Type=ThresholdType.Min, Threshold=bed_level_estimate - 0.1, Flag=QCFlags.Bad, Label=QCLabels.BelowRange))   # More than 0.1m below estimated bed level
    if (not np.isnan(bank_level_estimate)) and (gauge_parameter == 'Water Level [mAOD]'):
        range_checks_parameter.append(
            RangeCheckParams(Type=ThresholdType.Max, Threshold=bank_level_estimate + 0.5, Flag=QCFlags.Suspect, Label=QCLabels.AboveRange),  # More than 0.5m above estimated bank level
        )

    # Check every RangeCheckParams defined for the current parameter
    for range_check in range_checks_parameter:
        if range_check.Type == ThresholdType.Min:
            indices_failures = df_parameter[df_parameter[gauge_parameter] < range_check.Threshold].index
        elif range_check.Type == ThresholdType.Max:
            indices_failures = df_parameter[df_parameter[gauge_parameter] > range_check.Threshold].index
        else:
            indices_failures = []

        # Flag corresponding timesteps of quality check failures and store
        df_failure_flags = pd.DataFrame({'Quality Code': [QualityCode(range_check.Flag, range_check.Label)]}, index=indices_failures)
        dfs_failure_flags.append(df_failure_flags)

    # Return list of dataframes, each containing a timeseries with failure flags for a different range check
    return dfs_failure_flags


### Define all thresholds for rate-of-change checks

In [ ]:
# Define all rate-of-change thresholds

# Parameters to define a range check
RateOfChangeCheckParams = NamedTuple('RateOfChangeCheckParams', [
    ('Threshold', float),
    ('UnitTimeMinutes', float),
    ('GaugeFilter', List[str]),
    ('Flag', int),
    ('Label', QCLabels),
])

# All rate of change checks to be applied for each parameter type
rate_of_change_checks_all = {
    'Water Level [mAOD]': [
        RateOfChangeCheckParams(Threshold=1, UnitTimeMinutes=60, GaugeFilter=['Fluvial'], Flag=QCFlags.Suspect, Label=QCLabels.HighRateOfChange),    # Rate of change greater than 0.5m/hour in fluvial system
        RateOfChangeCheckParams(Threshold=0.05, UnitTimeMinutes=60, GaugeFilter=['Borehole'], Flag=QCFlags.Suspect, Label=QCLabels.HighRateOfChange)    # Rate of change greater than 0.05m/hour in borehole
    ],
    'Signal-to-noise ratio [dB]': [],
    'Water Depth [m]': [
        RateOfChangeCheckParams(Threshold=1, UnitTimeMinutes=60, GaugeFilter=['Fluvial'], Flag=QCFlags.Suspect, Label=QCLabels.HighRateOfChange)    # Rate of change greater than 0.5m/hour in fluvial system
    ],
    'Water Level [mm]': [
        RateOfChangeCheckParams(Threshold=100, UnitTimeMinutes=1, GaugeFilter=['Highway gully'], Flag=QCFlags.Suspect, Label=QCLabels.HighRateOfChange)    # Rate of change greater than 0.1m/minute in highway gully
    ],
    'Sewer Level [mm]': [
        RateOfChangeCheckParams(Threshold=100, UnitTimeMinutes=1, GaugeFilter=['Sewer'], Flag=QCFlags.Suspect, Label=QCLabels.HighRateOfChange)    # Rate of change greater than 0.1m/minute in sewer system
    ],
    'Depth To Water [m]': [
        RateOfChangeCheckParams(Threshold=0.05, UnitTimeMinutes=60, GaugeFilter=['Borehole'], Flag=QCFlags.Suspect, Label=QCLabels.HighRateOfChange)    # Rate of change greater than 0.05m/hour in borehole
    ],
    'Rainfall Depth [mm]': [
    ],
    'Water Temperature [°C]': [
        RateOfChangeCheckParams(Threshold=3, UnitTimeMinutes=60, GaugeFilter=['Fluvial', 'Borehole'], Flag=QCFlags.Suspect, Label=QCLabels.HighRateOfChange)    # Rate of change greater than 3°C / hour in water temperature
    ],
    'Air Temperature [°C]': [
        RateOfChangeCheckParams(Threshold=5, UnitTimeMinutes=60, GaugeFilter=['Fluvial', 'Borehole'], Flag=QCFlags.Suspect, Label=QCLabels.HighRateOfChange)    # Rate of change greater than 5°C / hour in air temperature
    ],
}

In [ ]:
# Function to check values are within all relevant range thresholds for a given parameter

def quality_check_parameter_rates_of_change(
    gauge_parameter: str,         # Name of gauge parameter
    gauge_type: str,              # Name of gauge type
    df_parameter: pd.DataFrame,   # Dataframe containing data for a single parameter
    rate_of_change_checks_all: dict,       # Dict containing list of RateOfChangeCheckParams for each gauge parameter
):
    # Initialise storage of all failure flags for current parameter (one dataframe per range check)
    dfs_failure_flags = []
    
    # Find details of rates of change to check for given parameter (list of RateOfChangeCheckParams)
    rate_of_change_checks_parameter = rate_of_change_checks_all[gauge_parameter].copy()

    # Check every RangeCheckParams defined for the current parameter, if from an appropriate gauge type
    for rate_of_change_check in rate_of_change_checks_parameter:
        if gauge_type in rate_of_change_check.GaugeFilter:

            # Calculate the rate of change per unit time
            dt_unit_time = df_parameter.copy().index.to_series().diff().dt.total_seconds() / (60 * rate_of_change_check.UnitTimeMinutes)
            dv = df_parameter[gauge_parameter].diff()
            rate_per_unit_time = dv / dt_unit_time

            # Find indices where rate of change per unit time exceeds threshold
            indices_failures = rate_per_unit_time[rate_per_unit_time.abs() > rate_of_change_check.Threshold].index
            
            # Flag corresponding timesteps of quality check failures and store
            df_failure_flags = pd.DataFrame({'Quality Code': [QualityCode(rate_of_change_check.Flag, rate_of_change_check.Label)]}, index=indices_failures)
            dfs_failure_flags.append(df_failure_flags)
    
    # Return list of dataframes, each containing a timeseries with failure flags for a different range check
    return dfs_failure_flags


### Define persistence checks

In [ ]:
# Define all persistence checks (static, non-zero measurements)

# Parameters to define a range check
PersistenceCheckParams = NamedTuple('PersistenceCheckParams', [
    ('ThresholdHours', float),
    ('GaugeFilter', List[str]),
    ('ZeroAllowed', bool),
    ('Flag', int),
    ('Label', QCLabels),
])

# Persistence check to be applied for each parameter type
persistence_checks_all = {
    'Water Level [mAOD]': [
        PersistenceCheckParams(ThresholdHours=6, GaugeFilter=['Fluvial'], ZeroAllowed=True, Flag=QCFlags.Suspect, Label=QCLabels.PersistentlyHigh),   # Static, non-zero measurement for over 6 hours in fluvial system
        PersistenceCheckParams(ThresholdHours=24, GaugeFilter=['Borehole'], ZeroAllowed=True, Flag=QCFlags.Suspect, Label=QCLabels.PersistentlyHigh)   # Static, non-zero measurement for over 6 hours in borehole
        ],
    'Signal-to-noise ratio [dB]': [],
    'Water Depth [m]': [
        PersistenceCheckParams(ThresholdHours=6, GaugeFilter=['Fluvial'], ZeroAllowed=True, Flag=QCFlags.Suspect, Label=QCLabels.PersistentlyHigh),   # Static, non-zero measurement for over 6 hours in fluvial system
        ],
    'Water Level [mm]': [],
    'Sewer Level [mm]': [
        PersistenceCheckParams(ThresholdHours=6, GaugeFilter=['Sewer'], ZeroAllowed=True, Flag=QCFlags.Suspect, Label=QCLabels.PersistentlyHigh),   # Static, non-zero measurement for over 6 hours
    ],
    'Depth To Water [m]': [
        PersistenceCheckParams(ThresholdHours=24, GaugeFilter=['Borehole'], ZeroAllowed=True, Flag=QCFlags.Suspect, Label=QCLabels.PersistentlyHigh),   # Static, non-zero measurement for over 24 hours in borehole
    ],
    'Rainfall Depth [mm]': [],
    'Water Temperature [°C]': [
        PersistenceCheckParams(ThresholdHours=6, GaugeFilter=['Fluvial', 'Borehole'], ZeroAllowed=False, Flag=QCFlags.Suspect, Label=QCLabels.PersistentlyHigh),   # Static, non-zero measurement for over 6 hours
    ],
    'Air Temperature [°C]': [
        PersistenceCheckParams(ThresholdHours=6, GaugeFilter=['Fluvial', 'Borehole'], ZeroAllowed=False, Flag=QCFlags.Suspect, Label=QCLabels.PersistentlyHigh),   # Static, non-zero measurement for over 6 hours
    ]
}

In [ ]:
# Function to check for persistent, values for a parameter

def quality_check_parameter_persistence(
    gauge_parameter: str,         # Name of gauge parameter
    gauge_type: str,              # Name of gauge type
    df_parameter: pd.DataFrame,   # Dataframe containing data for a single parameter
    persistence_checks_all: dict,       # Dict containing PersistenceCheckParams for each gauge parameter
):
    # Initialise storage of all failure flags for current parameter (one dataframe per range check)
    dfs_failure_flags = []

    # Find details of persistence check for given parameter (list of PersistenceCheckParams)
    persistence_checks_parameter = persistence_checks_all[gauge_parameter].copy()

    # Check every PersistenceCheckParams defined for the current parameter, if from an appropriate gauge type
    for persistence_check_parameter in persistence_checks_parameter:
        if gauge_type in persistence_check_parameter.GaugeFilter:

            # Identify where the value changes
            value_changes = df_parameter[gauge_parameter] != df_parameter[gauge_parameter].shift()

            # Assign IDs to runs of persistent readings
            run_id = value_changes.cumsum()
        
            # Compute run durations
            run_start = df_parameter.index.to_series().groupby(run_id).min()
            run_end = df_parameter.index.to_series().groupby(run_id).max()
            run_value = df_parameter[gauge_parameter].groupby(run_id).first()
            run_duration_hours = (run_end - run_start).dt.total_seconds() / 3600

            # Identify persistent reading runs exceeding specified duration
            if persistence_check_parameter.ZeroAllowed:
                stuck_run_ids = run_duration_hours[(run_duration_hours > persistence_check_parameter.ThresholdHours)].index
            else:
                stuck_run_ids = run_duration_hours[(run_duration_hours > persistence_check_parameter.ThresholdHours) & (run_value != 0)].index

            # Extract indices belonging to persistent reading runs
            mask = run_id.isin(stuck_run_ids)
            indices_failures = df_parameter.index[mask]

            # Flag corresponding timesteps of quality check failures and store
            df_failure_flags = pd.DataFrame({'Quality Code': [QualityCode(persistence_check_parameter.Flag, persistence_check_parameter.Label)]}, index=indices_failures)
            dfs_failure_flags.append(df_failure_flags)

            # # Return list of dataframes, each containing a timeseries with failure flags for a different persistence check
            # return df_failure_flags

    # Return list of dataframes, each containing a timeseries with failure flags for a different range check
    return dfs_failure_flags
    # return pd.DataFrame()


### Apply checks to all gauges and timeseries

In [ ]:
# Function to restructure quality dataframe for saving

def restructure_quality_df_to_save(df: pd.DataFrame):
    # Find all Quality Flag columns and extract parameter names
    flag_cols = {
        re.match(r'Quality Flag - (.+)', col).group(1): col
        for col in df.columns
        if col.startswith('Quality Flag - ')
    }

    # Find all Quality Label columns and extract parameter names
    label_cols = {
        re.match(r'Quality Labels - (.+)', col).group(1): col
        for col in df.columns
        if col.startswith('Quality Labels - ')
    }

    # Build new dataframe
    new_df = pd.DataFrame(index=df.index)

    for param in flag_cols.keys() & label_cols.keys():
        flag_col = flag_cols[param]
        label_col = label_cols[param]

        flag_series = pd.to_numeric(df[flag_col], errors='coerce')  # Ensure int type
        
        flag_str = flag_series.apply(
            lambda x: '' if pd.isna(x) else str(int(x))
        )

        label_str = df[label_col].astype(str).str.strip()

        new_df[f'Quality - {param}'] = (flag_str + label_str).str.strip()

    return new_df

In [ ]:
# Apply range checks to all gauges

# Initialise storage of quality summaries for each gauge (proportion and count of data points with each quality flag)
gauge_quality_proportion_summaries = {}
gauge_quality_count_summaries = {}

# Analyse each gauge
for index, row in df_index.iterrows():

    # Load data from gauge and convert to tz-aware datetime index
    location = row['locationName']
    gauge_type = row['gaugeType']
    data_path = row['dataPath']
    print(f'Analysing data from {location}')

    dfs_raw = get_gauge_data(
        data_type_option=DataTypeOptions.Raw,
        filter_gauge_names=[location]
    )

    # Get location-specific bank and bed levels for use in additional range-based RangeChecks, if available
    bed_level_estimate = row['bedLevelEstimate_mAOD']
    bank_level_estimate = row['bankLevelEstimate_mAOD']

    # Initialise storage of all failure flags for current gauge
    dfs_gauge_failure_flags = []

    # Apply quality checks to each parameter
    parameters = dfs_raw[location].keys()
    for parameter in parameters:
        
        # Extract data for parameter
        df_parameter = dfs_raw[location][parameter].dropna()

        # Initialise storage of all failure flags for current parameter alongside original data
        dfs_parameter_failure_flags = [df_parameter]

        # Apply range checks
        dfs_parameter_failure_flags_range_checks = quality_check_parameter_ranges(
            gauge_parameter=parameter.value,
            df_parameter=df_parameter,
            range_checks_all=range_checks_all,
            bed_level_estimate=bed_level_estimate,
            bank_level_estimate=bank_level_estimate
        )

        # Store range check failure flags
        dfs_parameter_failure_flags.extend(dfs_parameter_failure_flags_range_checks)

        # Apply rate of change checks
        dfs_parameter_failure_flags_rate_of_change_checks = quality_check_parameter_rates_of_change(
            gauge_parameter=parameter.value,
            gauge_type=gauge_type,
            df_parameter=df_parameter,
            rate_of_change_checks_all=rate_of_change_checks_all
        )

        # Store rate of change check failure flags
        dfs_parameter_failure_flags.extend(dfs_parameter_failure_flags_rate_of_change_checks)

        # Apply persistence checks
        df_parameter_failure_flags_persistence_checks = quality_check_parameter_persistence(
            gauge_parameter=parameter.value,
            gauge_type=gauge_type,
            df_parameter=df_parameter,
            persistence_checks_all=persistence_checks_all
        )

        # Store persistence check failure flags
        dfs_parameter_failure_flags.extend(df_parameter_failure_flags_persistence_checks)
      
        
        # Combine all failure flags for current parameter, along with original data, and calculate max (overall flag for parameter)
        df_parameter_with_quality = pd.concat(dfs_parameter_failure_flags, axis=1, join='outer')

        # Calculate overall quality flag for the parameter (worst across all tests). Default to 'good' if no failures.
        if (df_parameter_with_quality.columns == 'Quality Code').sum() > 1:
            df_parameter_with_quality[f'Quality Flag - {parameter.value}'] = df_parameter_with_quality['Quality Code'].apply(
                lambda row: max(
                    (quality_code.Flag.value
                    for quality_code in row
                    if pd.notna(quality_code)),
                    default=QCFlags.Good.value
                ),
                axis=1
            )
        elif (df_parameter_with_quality.columns == 'Quality Code').sum() == 1:
            df_parameter_with_quality[f'Quality Flag - {parameter.value}'] = df_parameter_with_quality['Quality Code'].apply(
                lambda row: row.Flag.value if pd.notna(row) else QCFlags.Good.value
            )
        else:
            raise Exception('No quality codes found')

        # Generate combined quality control label string identifying all failures.
        if (df_parameter_with_quality.columns == 'Quality Code').sum() > 1:
            df_parameter_with_quality[f'Quality Labels - {parameter.value}'] = df_parameter_with_quality['Quality Code'].apply(
                lambda row: ''.join(
                    sorted(
                        dict.fromkeys(
                            ''.join(
                                (quality_code.Label.value for quality_code in row if pd.notna(quality_code))
                            )
                        )
                    )
                ),
                axis=1
            )
        else:
            df_parameter_with_quality[f'Quality Labels - {parameter.value}'] = df_parameter_with_quality['Quality Code'].apply(
                lambda row: row.Label.value if pd.notna(row) else QCLabels.NA.value
            )

        # Store failure flags and labels for current parameter
        dfs_gauge_failure_flags.append(df_parameter_with_quality[[f'Quality Flag - {parameter.value}', f'Quality Labels - {parameter.value}']])

    # Combine all failure flags for current gauge (don't replicate original data)
    df_gauge_quality = pd.concat(dfs_gauge_failure_flags, axis=1, sort=True, join='outer')
    df_gauge_quality = df_gauge_quality.loc[:, df_gauge_quality.columns.str.startswith('Quality ')]

   
    # Save overall quality flags for each parameter
    if save_flag:
        save_path = f'{path_output_folder}/{data_path.replace('.csv', '_quality_flags.csv')}'
        # Make required folder if it doesn't exist
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        save_path = '\\\\?\\' + str(Path(save_path).resolve())
        restructure_quality_df_to_save(df_gauge_quality).to_csv(save_path)
    
    # Calculate and store stats for each quality check and overall quality flags for each timeseries
    gauge_quality_proportion_summaries[location] = {}
    gauge_quality_count_summaries[location] = {}
    for column in df_gauge_quality:
        quality_flag_proportions = df_gauge_quality[column].value_counts(normalize=True)
        gauge_quality_proportion_summaries[location][column] = quality_flag_proportions
        quality_flag_counts = df_gauge_quality[column].value_counts()
        gauge_quality_count_summaries[location][column] = quality_flag_counts